# Differential Equations — Session 26
## Section 6.1: Review of Power Series

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to define a power series and its center; determine a radius and interval of convergence; test endpoints; differentiate and integrate term by term; use the identity property; shift summation indices; and derive a recurrence relation from a differential equation.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Definitions and convergence |
| 18–35 min | Ratio test and endpoints |
| 35–52 min | Differentiation, integration, identity |
| 52–68 min | Index shifting |
| 68–84 min | First series solution |
| 84–90 min | Exit check |

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import jv, yv, iv, kv, eval_legendre, jn_zeros
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def polynomial_value(coefficients, x):
    x = np.asarray(x, dtype=float)
    total = np.zeros_like(x)
    for n, c in enumerate(coefficients):
        total += c*x**n
    return total

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 6.1-A — Power series

A power series centered at $a$ is

$$
\sum_{n=0}^{\infty}c_n(x-a)^n.
$$

Its $N$th partial sum is

$$
S_N(x)=\sum_{n=0}^{N}c_n(x-a)^n.
$$

### Theorem 6.1-B — Radius of convergence

There is a number $R\in[0,\infty]$ such that the series converges absolutely for $|x-a|<R$ and diverges for $|x-a|>R$. Finite endpoints must be tested separately.

### Theorem 6.1-C — Ratio test

If

$$
L=\lim_{n\to\infty}\left|\frac{a_{n+1}}{a_n}\right|,
$$

then the series converges absolutely for $L<1$, diverges for $L>1$, and the test is inconclusive for $L=1$.

### Theorem 6.1-D — Term-by-term operations

Inside the open interval of convergence,

$$
\frac{d}{dx}\sum_{n=0}^{\infty}c_n(x-a)^n
=
\sum_{n=1}^{\infty}nc_n(x-a)^{n-1},
$$

and

$$
\int\sum_{n=0}^{\infty}c_n(x-a)^n\,dx
=
C+\sum_{n=0}^{\infty}\frac{c_n}{n+1}(x-a)^{n+1}.
$$

### Theorem 6.1-E — Identity property

If a power series is identically zero on an open interval, then every coefficient is zero.

### Principle 6.1-F — Index alignment

Series can be combined only after their starting indices and powers are aligned.

### Classroom Checkpoint — Endpoint Warning

Why does the ratio test not finish an interval-of-convergence problem when the radius is finite?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Partial sums reveal convergence

The geometric series satisfies

$$
\sum_{n=0}^{\infty}x^n=\frac{1}{1-x},
\qquad |x|<1.
$$

In [ ]:
x = np.linspace(-0.95, 0.95, 600)
for N in [1, 3, 7, 15]:
    plt.plot(x, sum(x**n for n in range(N+1)), label=fr"$S_{N}$")
plt.plot(x, 1/(1-x), linewidth=3, linestyle="--", label=r"$1/(1-x)$")
plt.legend()
plt.title("Partial sums of the geometric series")
plt.show()

In [ ]:
def geometric_error(N=10):
    x = np.linspace(-0.98, 0.98, 700)
    error = np.abs(1/(1-x)-sum(x**n for n in range(N+1)))
    plt.semilogy(x, error)
    plt.xlabel("x")
    plt.ylabel("absolute error")
    plt.title(fr"Geometric-series error for $N={N}$")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(geometric_error, N=IntSlider(min=1, max=80, value=10))
else:
    geometric_error()

## 2. Endpoint analysis

For

$$
\sum_{n=1}^{\infty}\frac{(-1)^n(x-3)^n}{n2^n},
$$

the ratio test gives $|x-3|<2$, hence $1<x<5$. The endpoints must then be tested using ordinary numerical-series tests.

In [ ]:
center, R = 3, 2
plt.axvspan(center-R, center+R, alpha=0.2)
plt.axvline(center, linestyle="--", label="center")
plt.scatter([center-R, center+R], [0, 0], s=90)
plt.yticks([])
plt.xlabel("x")
plt.title("Center and radius of convergence")
plt.legend()
plt.show()

## 3. Differentiation and integration

From

$$
\frac{1}{1-x}=\sum_{n=0}^{\infty}x^n,
$$

we obtain

$$
\frac{1}{(1-x)^2}=\sum_{n=1}^{\infty}nx^{n-1}
$$

and

$$
-\ln(1-x)=\sum_{n=1}^{\infty}\frac{x^n}{n}.
$$

In [ ]:
x = np.linspace(-0.9, 0.9, 600)
for N in [2, 5, 10, 30]:
    plt.plot(x, sum(x**n/n for n in range(1, N+1)), label=f"N={N}")
plt.plot(x, -np.log(1-x), linestyle="--", linewidth=3, label="exact")
plt.legend()
plt.show()

## 4. Cauchy product

If $A(x)=\sum a_nx^n$ and $B(x)=\sum b_nx^n$, then

$$
A(x)B(x)=
\sum_{n=0}^{\infty}
\left(\sum_{k=0}^{n}a_kb_{n-k}\right)x^n.
$$

In [ ]:
x = sp.symbols("x")
display(sp.series(sp.exp(x)*sp.sin(x), x, 0, 9))

## 5. Shifting indices

To combine

$$
\sum_{n=2}^{\infty}n(n-1)c_nx^{n-2}
+
\sum_{n=0}^{\infty}c_nx^n,
$$

set $k=n-2$ in the first sum:

$$
\sum_{k=0}^{\infty}
\left[(k+2)(k+1)c_{k+2}+c_k\right]x^k.
$$

## 6. First series solution

For

$$
y'+y=0,
$$

assume $y=\sum c_nx^n$. Then

$$
(n+1)c_{n+1}+c_n=0,
$$

so

$$
c_{n+1}=-\frac{c_n}{n+1}
$$

and

$$
y=c_0e^{-x}.
$$

In [ ]:
def coeffs(c0=1, N=12):
    c = [c0]
    for n in range(N):
        c.append(-c[-1]/(n+1))
    return np.array(c)

x = np.linspace(-3, 3, 600)
for N in [2, 4, 8, 14]:
    plt.plot(x, polynomial_value(coeffs(1, N), x), label=f"N={N}")
plt.plot(x, np.exp(-x), linestyle="--", linewidth=3, label="exact")
plt.ylim(-2, 20)
plt.legend()
plt.show()

## Classroom Checkpoint — Exit Check

Rewrite

$$
\sum_{n=3}^{\infty}n(n-1)c_nx^{n-2}
$$

as a series beginning with $x^1$.

> Pause here. Let students commit to an answer before running the next cell.